# Nettoyage de la base d'apprentissage

In [47]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data_finale/base_apprentissage.csv")


# Variables avec beaucoup de valeurs manquantes
print("Colonnes avec plus de 30% de valeurs manquantes :")
taux_manquants = df.isnull().mean()
colonnes_vides = taux_manquants[taux_manquants > 0.30].sort_values(
    ascending=False
)
if not colonnes_vides.empty:
    for col, tx in colonnes_vides.items():
        print(f"   • {col} : {tx*100:.1f}% de valeurs manquantes")
else:
    print("   • Aucune colonne ne dépasse 30% de lignes vides.")

Colonnes avec plus de 30% de valeurs manquantes :
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 94.8% de valeurs manquantes
   • Performance_CS% : 93.0% de valeurs manquantes
   • Performance_Save% : 92.8% de valeurs manquantes
   • Performance_GA : 92.6% de valeurs manquantes
   • Performance_W : 92.6% de valeurs manquantes
   • Performance_Saves : 92.6% de valeurs manquantes
   • Performance_SoTA : 92.6% de valeurs manquantes
   • Performance_GA90 : 92.6% de valeurs manquantes
   • Penalty Kicks_PKatt : 92.6% de valeurs manquantes
   • Performance_D : 92.6% de valeurs manquantes
   • Performance_L : 92.6% de valeurs manquantes
   • Performance_CS : 92.6% de valeurs manquantes
   • Penalty Kicks_PKm : 92.6% de valeurs manquantes
   • Penalty Kicks_PKsv : 92.6% de valeurs manquantes
   • Penalty Kicks_PKA : 92.6% de valeurs manquantes
   • xg : 49.3% de valeurs manquantes
   • xa : 49.3% de valeurs

In [48]:
# Identification des outliers

print("\nDétection des Outliers (Méthode de l'Écart Interquartile - IQR) :")
# On nettoie rapidement l'âge pour éviter les bugs de détection
df["age"] = (
    df["age"].astype(str).str.split("-").str[0].apply(pd.to_numeric, errors="coerce")
)
df_num = df.select_dtypes(include=[np.number])

outliers_count = {}
for col in df_num.columns:
    Q1 = df_num[col].quantile(0.25)
    Q3 = df_num[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR

    # Compter le nombre de lignes hors limites
    nb_outliers = ((df_num[col] < borne_inf) | (df_num[col] > borne_sup)).sum()
    if nb_outliers > 0:
        outliers_count[col] = nb_outliers

# Tri pour afficher les colonnes avec le plus d'outliers (Top 10)
outliers_tries = sorted(outliers_count.items(), key=lambda x: x[1], reverse=True)
for col, nb in outliers_tries[:10]:
    print(f"   • {col} : {nb} valeurs extrêmes détectées")


Détection des Outliers (Méthode de l'Écart Interquartile - IQR) :
   • injury_musculaire_count : 3683 valeurs extrêmes détectées
   • injury_musculaire_nb_d : 3683 valeurs extrêmes détectées
   • injury_musculaire : 3683 valeurs extrêmes détectées
   • injury_musculaire_nb_m : 3542 valeurs extrêmes détectées
   • injury_minor_unknown_nb_d : 2877 valeurs extrêmes détectées
   • injury_minor_unknown_nb_m : 2643 valeurs extrêmes détectées
   • Performance_CrdR : 1906 valeurs extrêmes détectées
   • Team Success_+/- : 1845 valeurs extrêmes détectées
   • Performance_Crs : 1656 valeurs extrêmes détectées
   • injury_genou_count : 1652 valeurs extrêmes détectées


In [49]:
# Détermination des variables catégorielles potentiellement à normaliser

print("\nListe des variables catégorielles :")
cols_cat = df.select_dtypes(include=["object"]).columns.tolist()
for col in cols_cat:
    nb_uniques = df[col].nunique()
    print(f"   • {col} ({nb_uniques} modalités uniques)")


Liste des variables catégorielles :
   • league (5 modalités uniques)
   • team (137 modalités uniques)
   • player (5511 modalités uniques)
   • nation (129 modalités uniques)
   • pos (10 modalités uniques)
   • join_key (5509 modalités uniques)
   • match_method (17 modalités uniques)
   • date (259 modalités uniques)
   • date_of_birth (3373 modalités uniques)
   • name (4717 modalités uniques)
   • tm_join_key (4716 modalités uniques)
   • tm_join_key_full (4716 modalités uniques)


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_852\3206937943.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = df.select_dtypes(include=["object"]).columns.tolist()


In [50]:
# Détection du risque de fuite de données

print("\nAnalyse des risques de fuite de données (Data Leakage) :")

# On cherche les variables trop fortement corrélées (r > 0.95) à la cible ou suspectes par leur nom
cible = "market_value_in_eur"
if cible in df.columns:
    correlations = df_num.corr()[cible].abs()
    leakage_stats = correlations[
        (correlations > 0.90) & (correlations.index != cible)
    ]

    if not leakage_stats.empty:
        print("   🔴 ALERTE : Variables suspectes (Corrélation quasi-parfaite) :")
        for col, val in leakage_stats.items():
            print(f"     - {col} (r = {val:.4f}) -> Risque fort de fuite !")
    else:
        print(
            "   • Aucun indicateur numérique n'affiche une corrélation suspecte proche de 1."
        )

    # Alerte sur la présence de mots-clés interdits dans les noms de colonnes
    mots_interdits = ["rank", "classement", "mv_", "log_"]
    suspects_nom = [
        col
        for col in df.columns
        if any(mot in col.lower() for mot in mots_interdits)
    ]
    if suspects_nom:
        print(
            f"   ATTENTION : Noms de colonnes à surveiller (indices de classement/target modifiée) : {suspects_nom}"
        )
else:
    print("   • Cible 'market_value_in_eur' non trouvée pour l'analyse.")


Analyse des risques de fuite de données (Data Leakage) :
   • Aucun indicateur numérique n'affiche une corrélation suspecte proche de 1.


In [53]:
# Doublons joueurs + saison + club
print("\nVérification des doublons par triplet [Joueur + Saison + Club] :")

# Identification des colonnes clés
col_joueur = "player" if "player" in df.columns else None
col_saison = "season" if "season" in df.columns else None
col_club = "team" if "team" in df.columns else None

if col_joueur and col_saison and col_club:
    # On cherche les doublons en incluant le club dans le subset
    nb_doublons_stricts = df.duplicated(
        subset=[col_joueur, col_saison, col_club]
    ).sum()

    if nb_doublons_stricts > 0:
        print(
            f"   Attention : {nb_doublons_stricts} lignes sont des doublons stricts (Même joueur, même saison, même club) !"
        )
        print("   Exemple de lignes concernées :")
        print(
            df[
                df.duplicated(
                    subset=[col_joueur, col_saison, col_club], keep=False
                )
            ][[col_joueur, col_saison, col_club]].head(6)
        )
    else:
        print(
            "   Parfait ! Aucune ligne en doublon pour un même joueur, une même saison et un même club."
        )
        print(
            "   Cela prouve que vos 1656 doublons précédents étaient uniquement dus aux transferts de mi-saison (mercato)."
        )
else:
    print(
        "   • Impossible de vérifier : les colonnes 'player', 'season' ou 'team' sont manquantes."
    )


Vérification des doublons par triplet [Joueur + Saison + Club] :
   Attention : 787 lignes sont des doublons stricts (Même joueur, même saison, même club) !
   Exemple de lignes concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea


In [52]:
import numpy as np
import pandas as pd

print(f"Format initial de la base : {df.shape}")

# Ajouter un compteur de valeurs manquantes
df["nb_manquants_ligne"] = df.isnull().sum(axis=1)


# On trie par Joueur, Saison, Club, ET par le nombre de manquants CROISSANT
# Ainsi, pour un même joueur/saison/club, la ligne avec le MOINS de NaN sera en haut
df_tri_tech = df.sort_values(
    by=["player", "season", "team", "nb_manquants_ligne"],
    ascending=[True, True, True, True],
)

# On supprime en gardant la première (donc celle qui a le moins de manquants)
df_sans_doublons_tech = df_tri_tech.drop_duplicates(
    subset=["player", "season", "team"], keep="first"
)

print(
    f"Format après suppression des doublons techniques (ligne la plus complète gardée) : {df_sans_doublons_tech.shape}"
)

df_sans_doublons_tech.to_csv("../data_finale/base_apprentissage.csv", index=False)

Format initial de la base : (17122, 129)
Format après suppression des doublons techniques (ligne la plus complète gardée) : (16335, 130)


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_852\3829890348.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["nb_manquants_ligne"] = df.isnull().sum(axis=1)


In [58]:
# Doublons joueurs + saison (liés au mercato)

# On recharge le nouveau dataset
df = pd.read_csv("../data_finale/base_apprentissage.csv")

print("\nVérification des doublons par couple [Joueur + Saison] liés au mercato :")
# Identification automatique des colonnes de clé
col_joueur = "player" if "player" in df.columns else None
col_saison = "season" if "season" in df.columns else None

if col_joueur and col_saison:
    nb_doublons = df.duplicated(subset=[col_joueur, col_saison]).sum()
    if nb_doublons > 0:
        print(
            f"   Attention : {nb_doublons} lignes sont des doublons stricts pour le même joueur lors de la même saison !"
        )
        print("   Exemple de lignes concernées :")
        print(
            df[df.duplicated(subset=[col_joueur, col_saison], keep=False)][
                [col_joueur, col_saison]
            ].head(4)
        )
else:
    print(
        "   • Impossible de vérifier : les colonnes 'player' ou 'season' sont manquantes."
    )


Vérification des doublons par couple [Joueur + Saison] liés au mercato :
   Attention : 869 lignes sont des doublons stricts pour le même joueur lors de la même saison !
   Exemple de lignes concernées :
          player  season
40  Aarón Martín    2021
41  Aarón Martín    2021
49  Abakar Sylla    2526
50  Abakar Sylla    2526


In [ ]:
# TODO : gérer les doublons de mercato

## **Les joueurs orphelins**

In [1]:
import pandas as pd

still_missing = pd.read_csv(r'..\data_finale\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\soccerdata.csv', encoding='utf-8-sig')

In [2]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 685
Taux joueurs orphelins : 11.06%


In [3]:
orphelins_par_saison = (
    still_missing
    .groupby('season')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season
2021     12
2122     36
2223     48
2324     81
2425     49
2526    552
dtype: int64


In [4]:
top_orphelins = (
    still_missing['player']
    .value_counts()
    .head(20)
)

print(top_orphelins)

player
Babis Lykogiannis    6
Hianga'a Mbock       5
Jonathan Rowe        4
Álex Jiménez         4
Yellu Santiago       4
Karl Etta Eyong      4
Ben Gannon-Doak      3
Kosta Nedeljković    3
Manuel Ugarte        3
Benjamin Šeško       3
Marc Pubill          3
Abde Rebbach         3
Bertuğ Yıldırım      3
Abdulai Juma Bah     3
Ismaël Boura         3
Hákon Haraldsson     3
Merveille Papela     3
Keke Topp            3
Dženan Pejčinović    3
Domen Črnigoj        3
Name: count, dtype: int64


## Nettoyage